In [9]:
# Build ALFIN raw data from source files

import os
import pandas as pd
import numpy as np
import dotenv

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

dotenv.load_dotenv(dotenv.find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

ALFIN_PATH = os.path.join(RAW_DATA_PATH, "alfin")

OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data/alfin_raw.parquet")
GEO_OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data/alfin_raw_geo.parquet")

COLMAP_2012_2016 = {
    "ID": (1, 14), 
    "STATE_FIPS": (1, 2),
    "UNIT_TYPE": (3, 3),
    "COUNTY_FIPS": (4, 6),
    "UNIT_ID": (7, 9),
    "PART_FLAG": (10, 14),  # should be 00000 to indicate unit is not part of another gov't unit
    "ITEM_CODE": (15, 17),
    "AMOUNT": (18, 29),  # amount is in thousands
    "YEAR": (30, 33),
    "IMPUTATION_FLAG": (34, 34)
}

GEO_COLMAP_2012_2016 = {
    "ID": (1, 14),
    "NAME": (15, 78),
    "COUNTY_NAME": (79, 113),
    "FIPS_STATE": (114, 115),
    "FIPS_COUNTY": (116, 118),
    "FIPS_PLACE": (119, 123),
    "POPULATION": (124, 132),
    "POPULATION_YEAR": (133, 134),
    "ENROLLMENT": (135, 141),
    "ENROLLMENT_YEAR": (142, 143),
    "FUNCTION_CODE": (114, 145),
    "SCHOOL_LEVEL_CODE": (146, 147),
    "FISCAL_YEAR_ENDING": (148, 151),
    "SURVEY_YEAR": (152, 153)
}

COLMAP_2017_2023 = {
    "ID": (1, 12),
    "STATE_FIPS": (1, 2),
    "UNIT_TYPE": (3, 3),
    "COUNTY_FIPS": (4, 6),
    "UNIT_ID": (7, 12),
    "ITEM_CODE": (13, 15),
    "AMOUNT": (16, 27),  # amount is in thousands
    "YEAR": (28, 31), 
    "IMPUTATION_FLAG": (32, 32)
}

GEO_COLMAP_2017_2023 = {
    "ID": (1, 12),
    "NAME": (13, 76),
    "COUNTY_NAME": (77, 111),
    "FIPS_PLACE": (112, 116),
    "POPULATION": (117, 125),
    "POPULATION_YEAR": (126, 127),
    "ENROLLMENT": (128, 134),
    "ENROLLMENT_YEAR": (135, 136),
    "FUNCTION_CODE": (137, 138),
    "SCHOOL_LEVEL_CODE": (139, 140),
    "FISCAL_YEAR_ENDING": (141, 144),
    "SURVEY_YEAR": (145, 146)
}

ALFIN_FILES = {
    "2012": {
        "filename": "2012/2012FinEstDAT_10162019modp_pu.txt",
        "geo_filename": "2012/Fin_GID_2012.txt",
        "columns": COLMAP_2012_2016,
        "geo_columns": GEO_COLMAP_2012_2016
    },
    "2013": {
        "filename": "2013/2013FinEstDAT_10162019modp_pu.txt",
        "geo_filename": "2013/Fin_GID_2013.txt",
        "columns": COLMAP_2012_2016,
        "geo_columns": GEO_COLMAP_2012_2016
    },
    "2014": {
        "filename": "2014/2014FinEstDAT_10162019modp_pu.txt",
        "geo_filename": "2014/Fin_GID_2014.txt",
        "columns": COLMAP_2012_2016,
        "geo_columns": GEO_COLMAP_2012_2016
    },
    "2015": {
        "filename": "2015/2015FinEstDAT_10162019modp_pu.txt",
        "geo_filename": "2015/Fin_GID_2015.txt",
        "columns": COLMAP_2012_2016,
        "geo_columns": GEO_COLMAP_2012_2016
    },
    "2016": {
        "filename": "2016/2016FinEstDAT_10162019modp_pu.txt",
        "geo_filename": "2016/Fin_GID_2016.txt",
        "columns": COLMAP_2012_2016,
        "geo_columns": GEO_COLMAP_2012_2016
    },
    "2017": {
        "filename": "2017/2017FinEstDAT_09202024modp_pu.txt",
        "geo_filename": "2017/Fin_PID_2017.txt",
        "columns": COLMAP_2017_2023,
        "geo_columns": GEO_COLMAP_2017_2023
    },
    "2018": {
        "filename": "2018/2018FinEstDAT_09202024modp_pu.txt",
        "geo_filename": "2018/Fin_PID_2018.txt",
        "columns": COLMAP_2017_2023,
        "geo_columns": GEO_COLMAP_2017_2023
    },
    "2019": {
        "filename": "2019/2019FinEstDAT_09202024modp_pu.txt",
        "geo_filename": "2019/Fin_PID_2019.txt",
        "columns": COLMAP_2017_2023,
        "geo_columns": GEO_COLMAP_2017_2023
    },
    "2020": {
        "filename": "2020/2020FinEstDAT_09202024modp_pu.txt",
        "geo_filename": "2020/Fin_PID_2020.txt",
        "columns": COLMAP_2017_2023,
        "geo_columns": GEO_COLMAP_2017_2023
    },
    "2021": {
        "filename": "2021/2021FinEstDAT_09202024modp_pu.txt",
        "geo_filename": "2021/Fin_PID_2021.txt",
        "columns": COLMAP_2017_2023,
        "geo_columns": GEO_COLMAP_2017_2023
    },
    "2022": {
        "filename": "2022/2022_Individual_Unit_File/2022FinEstDAT_06052025modp_pu.txt",
        "geo_filename": "2022/2022_Individual_Unit_File/Fin_PID_2022.txt",
        "columns": COLMAP_2017_2023,
        "geo_columns": GEO_COLMAP_2017_2023
    },
    "2023": {
        "filename": "2023/2023_Individual_Unit_Files/2023FinEstDAT_06052025modp_pu.txt",
        "geo_filename": "2023/2023_Individual_Unit_Files/Fin_PID_2023.txt",
        "columns": COLMAP_2017_2023,
        "geo_columns": GEO_COLMAP_2017_2023
    }
}    



In [10]:
# item level data files

df = []
for year, year_dict in ALFIN_FILES.items():
    file_path = os.path.join(ALFIN_PATH, year_dict["filename"])
    columns = year_dict["columns"]
    my_df = []
    with open(file_path, 'r') as f:
        for line in f.readlines():
            row = {}
            for k, v in columns.items():
                start = v[0]-1
                end = v[1]
                row[k] = line[start:end].strip()
            my_df.append(row)

    my_df = pd.DataFrame(my_df)
    my_df['AMOUNT'] = my_df['AMOUNT'].astype(float)
    my_df['YEAR'] = my_df['YEAR'].astype(int)
    df.append(my_df)

df = pd.concat(df).reset_index(drop=True)

df.to_parquet(OUTPUT_FILEPATH)


In [11]:
# geographic identification files

geo_df = []
for year, year_dict in ALFIN_FILES.items():
    file_path = os.path.join(ALFIN_PATH, year_dict["geo_filename"])
    columns = year_dict["geo_columns"]
    my_df = []
    with open(file_path, 'r') as f:
        for line in f.readlines():
            row = {}
            for k, v in columns.items():
                start = v[0]-1
                end = v[1]
                row[k] = line[start:end].strip()
            my_df.append(row)

    my_df = pd.DataFrame(my_df)
    my_df['YEAR'] = int(year)
    geo_df.append(my_df)

geo_df = pd.concat(geo_df).reset_index(drop=True)

geo_df.to_parquet(GEO_OUTPUT_FILEPATH)